# V18 BCR/TCR Donor-Level IT-Oriented Analysis
**Date:** 2026-03-14
**Component:** C12 — BCR/TCR Repertoire Analysis
**Method:** Donor-level Mann-Whitney U (consistent with C3-C10 pipeline)
**Key questions:**
1. Is plasmaB_c01-SDC1 collapse donor-consistent at NL→IT?
2. B cell subcluster redistribution — which are IT-specific?
3. BCR clonality, isotype shift, V-gene usage — tissue-separated donor-level
4. TCR clonality — tissue-separated donor-level
5. Pattern classification: IT-specific vs chronic-persistent

In [1]:
# ============================================================
# CELL 0: GPU SETUP & VERIFICATION
# ============================================================
!pip install scanpy anndata matplotlib seaborn scipy -q

import subprocess
import sys

print("=" * 70)
print("  V18 C3: GPU ENVIRONMENT SETUP")
print("=" * 70)

# GPU detection
try:
    result = subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,compute_cap',
                            '--format=csv,noheader'], capture_output=True, text=True)
    gpu_info = result.stdout.strip()
    print(f"✅ GPU detected: {gpu_info}")
except:
    print("⚠️ No GPU detected — will use CPU fallback")

# Install CuPy for GPU acceleration
try:
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} ready")
    print(f"   GPU memory: {cp.cuda.Device(0).mem_info[1] / 1e9:.1f} GB total")
except ImportError:
    print("📦 Installing CuPy...")
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'cupy-cuda12x', '-q'])
    import cupy as cp
    print(f"✅ CuPy {cp.__version__} installed")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 66.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.6/176.6 kB 21.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 284.1/284.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 82.9 MB/s eta 0:00:00
  V18 C3: GPU ENVIRONMENT SETUP
✅ GPU detected: NVIDIA A100-SXM4-80GB, 81920 MiB, 8.0
✅ CuPy 14.0.1 ready
   GPU memory: 85.1 GB total


In [20]:
# Cell 1: Setup
from google.colab import drive
drive.mount('/content/drive')

import scanpy as sc
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu
import warnings
warnings.filterwarnings('ignore')

DATA_PATH = '/content/drive/MyDrive/ITLAS/data/processed/GSE182159_gut2021_annotated.h5ad'
SAVE_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2/BCR_TCR'
import os; os.makedirs(SAVE_DIR, exist_ok=True)

adata = sc.read_h5ad(DATA_PATH, backed='r')
obs = adata.obs.copy()
obs['donor'] = obs['sample'].astype(str).str.split('_').str[1]

def safe_clonality(clone_counts):
    nu = len(clone_counts)
    nt = clone_counts.sum()
    if nu <= 0 or nt <= 0: return 0.0
    if nu == 1: return 1.0 if nt > 1 else 0.0
    fr = clone_counts.values / nt
    fr = fr[fr > 0]
    ent = -np.sum(fr * np.log2(fr))
    return 1 - (ent / np.log2(nu)) if np.log2(nu) > 0 else 0.0

def mw_test(nl_vals, it_vals, label=''):
    """Mann-Whitney with consistency count."""
    nl_v = nl_vals.dropna()
    it_v = it_vals.dropna()
    if len(nl_v) < 2 or len(it_v) < 2:
        return None
    stat, p = mannwhitneyu(nl_v, it_v, alternative='two-sided')
    nl_m, it_m = nl_v.mean(), it_v.mean()
    d = '↑' if it_m > nl_m else '↓'
    pct = ((it_m - nl_m) / nl_m * 100) if nl_m != 0 else float('inf')
    # Donor-pair consistency
    pairs_total = 0; pairs_consistent = 0
    for nv in nl_v:
        for iv in it_v:
            pairs_total += 1
            if (it_m > nl_m and iv > nv) or (it_m <= nl_m and iv <= nv):
                pairs_consistent += 1
    consistency = f'{pairs_consistent}/{pairs_total}'
    sig = '★' if p < 0.05 else '†' if p < 0.10 else ' '
    return {'metric': label, 'NL_mean': nl_m, 'IT_mean': it_m,
            'direction': d, 'pct_change': pct, 'p_value': p,
            'consistency': consistency, 'sig': sig}

print(f'Total cells: {len(obs):,}')
print(f'Donors: {obs["donor"].nunique()}')
print('Setup complete.')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Total cells: 243,000
Donors: 23
Setup complete.


In [21]:
# Cell 2: B/PlasmaB SUBCLUSTER PROPORTIONS — Donor-Level
# Key: proportion of each subcluster within total B+PlasmaB cells per donor per tissue
print('='*70)
print('SECTION A: B/PlasmaB Subcluster Proportions (Donor-Level)')
print('='*70)

b_plasma = obs[obs['major_lineage'].isin(['B', 'PlasmaB'])].copy()
subclusters = sorted(b_plasma['gut2021_subcluster_v2'].unique())
print(f'B/PlasmaB subclusters: {subclusters}')
print(f'Total B/PlasmaB cells: {len(b_plasma):,}')

# Calculate donor-level proportions
prop_rows = []
for (stage, tissue, donor), grp in b_plasma.groupby(['Stage', 'tissue', 'donor'], observed=True):
    total = len(grp)
    if total < 5:  # skip donors with very few B cells
        continue
    sc_counts = grp['gut2021_subcluster_v2'].value_counts()
    for sc in subclusters:
        count = sc_counts.get(sc, 0)
        prop_rows.append({
            'Stage': stage, 'tissue': tissue, 'donor': donor,
            'subcluster': sc, 'count': count, 'total_bp': total,
            'proportion': count / total * 100
        })

prop_df = pd.DataFrame(prop_rows)
print(f'Donor-level proportion records: {len(prop_df)}')

# Mann-Whitney NL→IT and NL→IA for each subcluster × tissue
results_a = []
for tissue_val in ['Liver', 'Blood']:
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()}: B/PlasmaB Subcluster Proportions NL→IT')
    print(f'{"─"*70}')
    for sc in subclusters:
        sub = prop_df[(prop_df['tissue'] == tissue_val) & (prop_df['subcluster'] == sc)]
        nl = sub[sub['Stage'] == 'NL']['proportion']
        it = sub[sub['Stage'] == 'IT']['proportion']
        ia = sub[sub['Stage'] == 'IA']['proportion']

        # NL→IT
        r_it = mw_test(nl, it, f'{sc}')
        # NL→IA
        r_ia = mw_test(nl, ia, f'{sc}')

        if r_it:
            # Pattern classification
            it_sig = r_it['p_value'] < 0.05
            ia_sig = r_ia['p_value'] < 0.05 if r_ia else False
            if it_sig and not ia_sig:
                pattern = 'IT-specific'
            elif it_sig and ia_sig:
                pattern = 'Chronic-persistent'
            elif not it_sig and ia_sig:
                pattern = 'IA-emergent'
            else:
                pattern = 'NS'

            r_it['tissue'] = tissue_val
            r_it['NL_IA_p'] = r_ia['p_value'] if r_ia else np.nan
            r_it['pattern'] = pattern
            results_a.append(r_it)

            ia_p_str = f'p={r_ia["p_value"]:.4f}' if r_ia else 'N/A'
            print(f'  {r_it["sig"]} {sc}: NL={r_it["NL_mean"]:.1f}% → IT={r_it["IT_mean"]:.1f}% '
                  f'({r_it["direction"]}{abs(r_it["pct_change"]):.1f}%) p={r_it["p_value"]:.4f} '
                  f'[{r_it["consistency"]}] | NL→IA {ia_p_str} | {pattern}')

results_a_df = pd.DataFrame(results_a)
results_a_df.to_csv(f'{SAVE_DIR}/A_subcluster_proportions_MW.csv', index=False)
print(f'\nSaved: A_subcluster_proportions_MW.csv')

SECTION A: B/PlasmaB Subcluster Proportions (Donor-Level)
B/PlasmaB subclusters: ['B_c01-IGHD', 'B_c02-STX16', 'B_c03-CD1C', 'B_c04-COCH', 'B_c05-TCL1A', 'B_c06-CD70', 'B_c07-FCRL5', 'plasmaB_c01-SDC1', 'plasmaB_c02-CD52', 'plasmaB_c03-MKI67']
Total B/PlasmaB cells: 23,408
Donor-level proportion records: 420

──────────────────────────────────────────────────────────────────────
LIVER: B/PlasmaB Subcluster Proportions NL→IT
──────────────────────────────────────────────────────────────────────
    B_c01-IGHD: NL=0.5% → IT=1.5% (↑214.4%) p=0.1908 [21/30] | NL→IA p=0.6321 | NS
    B_c02-STX16: NL=0.4% → IT=0.9% (↑110.9%) p=0.2622 [20/30] | NL→IA p=0.7739 | NS
    B_c03-CD1C: NL=0.2% → IT=1.4% (↑511.8%) p=0.1036 [22/30] | NL→IA p=0.0195 | IA-emergent
  ★ B_c04-COCH: NL=1.3% → IT=7.1% (↑437.8%) p=0.0135 [29/30] | NL→IA p=0.0222 | Chronic-persistent
    B_c05-TCL1A: NL=23.2% → IT=12.8% (↓44.9%) p=0.6623 [18/30] | NL→IA p=0.7922 | NS
    B_c06-CD70: NL=17.3% → IT=15.2% (↓11.9%) p=0.7922 [17/

In [22]:
# Cell 3: PlasmaB/B RATIO — Donor-Level (differentiation efficiency)
print('='*70)
print('SECTION B: PlasmaB/B Ratio = Differentiation Efficiency')
print('='*70)

ratio_rows = []
for (stage, tissue, donor), grp in b_plasma.groupby(['Stage', 'tissue', 'donor'], observed=True):
    n_b = (grp['major_lineage'] == 'B').sum()
    n_pb = (grp['major_lineage'] == 'PlasmaB').sum()
    n_sdc1 = (grp['gut2021_subcluster_v2'] == 'plasmaB_c01-SDC1').sum()
    total = n_b + n_pb
    if total < 5:
        continue
    ratio_rows.append({
        'Stage': stage, 'tissue': tissue, 'donor': donor,
        'n_B': n_b, 'n_PlasmaB': n_pb, 'n_SDC1': n_sdc1,
        'total_BP': total,
        'PlasmaB_ratio': n_pb / total * 100,
        'SDC1_ratio': n_sdc1 / total * 100,
        'SDC1_of_PlasmaB': n_sdc1 / n_pb * 100 if n_pb > 0 else 0,
    })

ratio_df = pd.DataFrame(ratio_rows)

results_b = []
for tissue_val in ['Liver', 'Blood']:
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()}')
    print(f'{"─"*70}')
    t = ratio_df[ratio_df['tissue'] == tissue_val]
    for stage in ['NL', 'IT', 'IA', 'AR', 'CR']:
        s = t[t['Stage'] == stage]
        if len(s) == 0:
            continue
        print(f'  {stage} ({len(s)} donors): PlasmaB ratio={s.PlasmaB_ratio.mean():.1f}%, '
              f'SDC1 ratio={s.SDC1_ratio.mean():.1f}%, '
              f'SDC1/PlasmaB={s.SDC1_of_PlasmaB.mean():.1f}%')

    # Mann-Whitney
    for metric in ['PlasmaB_ratio', 'SDC1_ratio', 'SDC1_of_PlasmaB']:
        nl = t[t['Stage'] == 'NL'][metric]
        it = t[t['Stage'] == 'IT'][metric]
        ia = t[t['Stage'] == 'IA'][metric]
        r_it = mw_test(nl, it, f'{metric}')
        r_ia = mw_test(nl, ia, f'{metric}')
        if r_it:
            it_sig = r_it['p_value'] < 0.05
            ia_sig = r_ia['p_value'] < 0.05 if r_ia else False
            pattern = 'IT-specific' if it_sig and not ia_sig else \
                      'Chronic' if it_sig and ia_sig else \
                      'IA-emergent' if not it_sig and ia_sig else 'NS'
            r_it['tissue'] = tissue_val; r_it['pattern'] = pattern
            results_b.append(r_it)
            ia_p = r_ia['p_value'] if r_ia else np.nan
            print(f'  {r_it["sig"]} {metric}: NL={r_it["NL_mean"]:.2f}→IT={r_it["IT_mean"]:.2f} '
                  f'({r_it["direction"]}{abs(r_it["pct_change"]):.1f}%) p={r_it["p_value"]:.4f} '
                  f'[{r_it["consistency"]}] NL→IA p={ia_p:.4f} | {pattern}')

pd.DataFrame(results_b).to_csv(f'{SAVE_DIR}/B_differentiation_ratio_MW.csv', index=False)
ratio_df.to_csv(f'{SAVE_DIR}/B_donor_level_ratios.csv', index=False)
print(f'\nSaved: B_differentiation_ratio_MW.csv, B_donor_level_ratios.csv')

SECTION B: PlasmaB/B Ratio = Differentiation Efficiency

──────────────────────────────────────────────────────────────────────
LIVER
──────────────────────────────────────────────────────────────────────
  NL (6 donors): PlasmaB ratio=44.1%, SDC1 ratio=32.5%, SDC1/PlasmaB=66.9%
  IT (5 donors): PlasmaB ratio=32.8%, SDC1 ratio=26.0%, SDC1/PlasmaB=76.8%
  IA (5 donors): PlasmaB ratio=10.5%, SDC1 ratio=5.0%, SDC1/PlasmaB=48.7%
  AR (3 donors): PlasmaB ratio=20.1%, SDC1 ratio=7.8%, SDC1/PlasmaB=43.4%
  CR (3 donors): PlasmaB ratio=16.4%, SDC1 ratio=6.0%, SDC1/PlasmaB=37.3%
    PlasmaB_ratio: NL=44.09→IT=32.85 (↓25.5%) p=0.6623 [18/30] NL→IA p=0.3290 | NS
    SDC1_ratio: NL=32.45→IT=26.00 (↓19.9%) p=0.9307 [16/30] NL→IA p=0.1775 | NS
    SDC1_of_PlasmaB: NL=66.90→IT=76.81 (↑14.8%) p=0.2468 [22/30] NL→IA p=0.4286 | NS

──────────────────────────────────────────────────────────────────────
BLOOD
──────────────────────────────────────────────────────────────────────
  NL (5 donors): PlasmaB r

In [23]:
# Cell 4: BCR REPERTOIRE — Donor-Level (clonality, isotype, V-gene)
print('='*70)
print('SECTION C: BCR Repertoire Donor-Level Metrics')
print('='*70)

def donor_bcr_full(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val]
    rows = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n = len(grp)
        bcr = grp[grp['BCR_clone.id'].notna()]
        nb = len(bcr)
        if nb == 0:
            rows.append({'Stage':stage,'donor':donor,'tissue':tissue_val,
                         'n_bcr':0,'pct_bcr':0,
                         'clonality':np.nan,'pct_singleton':np.nan,
                         'pct_IgM':np.nan,'pct_IgG':np.nan,'pct_IgA':np.nan,'pct_IgD':np.nan,
                         'pct_switched':np.nan,'top_clone':0,
                         'pct_IGHV3_23':np.nan,'n_unique_vgenes':0})
            continue
        cc = bcr['BCR_clone.id'].value_counts()
        clon = safe_clonality(cc)
        nu = len(cc); ns = (cc==1).sum()
        iso = bcr['BCR_CType'].value_counts(); it = iso.sum()
        igm = iso.get('IGHM',0)/it*100; igg = iso.get('IGHG',0)/it*100
        iga = iso.get('IGHA',0)/it*100; igd = iso.get('IGHD',0)/it*100
        vg = bcr['BCR_v_gene'].value_counts()
        pct_v3_23 = vg.get('IGHV3-23',0)/nb*100
        rows.append({'Stage':stage,'donor':donor,'tissue':tissue_val,
                     'n_bcr':nb,'pct_bcr':nb/n*100,
                     'clonality':clon,'pct_singleton':ns/nu*100,
                     'pct_IgM':igm,'pct_IgG':igg,'pct_IgA':iga,'pct_IgD':igd,
                     'pct_switched':igg+iga,'top_clone':cc.max(),
                     'pct_IGHV3_23':pct_v3_23,'n_unique_vgenes':len(vg)})
    return pd.DataFrame(rows)

bcr_results = []
for tissue_val in ['Liver','Blood']:
    df = donor_bcr_full(obs, tissue_val)
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — BCR Repertoire')
    print(f'{"─"*70}')
    for stage in ['NL','IT','IA','AR','CR']:
        s = df[(df['Stage']==stage) & (df['n_bcr']>0)]
        if len(s)==0:
            print(f'  {stage}: no BCR'); continue
        print(f'  {stage} ({len(s)}d): BCR={s.n_bcr.mean():.0f}, '
              f'clon={s.clonality.mean():.4f}, sing={s.pct_singleton.mean():.1f}%, '
              f'IgM={s.pct_IgM.mean():.1f}% IgG={s.pct_IgG.mean():.1f}% '
              f'IgA={s.pct_IgA.mean():.1f}% IgD={s.pct_IgD.mean():.1f}% '
              f'sw={s.pct_switched.mean():.1f}% V3-23={s.pct_IGHV3_23.mean():.1f}%')

    # Mann-Whitney NL→IT and NL→IA
    nl = df[(df['Stage']=='NL') & (df['n_bcr']>0)]
    it = df[(df['Stage']=='IT') & (df['n_bcr']>0)]
    ia = df[(df['Stage']=='IA') & (df['n_bcr']>0)]
    if len(nl)>=2 and len(it)>=2:
        print(f'\n  Mann-Whitney NL→IT ({tissue_val}):')
        for m in ['clonality','pct_singleton','pct_IgM','pct_IgG','pct_IgA','pct_IgD',
                  'pct_switched','pct_bcr','top_clone','pct_IGHV3_23']:
            r_it = mw_test(nl[m], it[m], m)
            r_ia = mw_test(nl[m], ia[m], m) if len(ia)>=2 else None
            if r_it:
                it_sig = r_it['p_value']<0.05
                ia_sig = r_ia['p_value']<0.05 if r_ia else False
                pattern = 'IT-spec' if it_sig and not ia_sig else \
                          'Chronic' if it_sig and ia_sig else \
                          'IA-emrg' if not it_sig and ia_sig else 'NS'
                ia_p = f'p={r_ia["p_value"]:.4f}' if r_ia else 'N/A'
                r_it['tissue']=tissue_val; r_it['pattern']=pattern
                bcr_results.append(r_it)
                print(f'    {r_it["sig"]} {m}: {r_it["NL_mean"]:.2f}→{r_it["IT_mean"]:.2f} '
                      f'({r_it["direction"]}{abs(r_it["pct_change"]):.1f}%) p={r_it["p_value"]:.4f} '
                      f'[{r_it["consistency"]}] | IA:{ia_p} | {pattern}')

pd.DataFrame(bcr_results).to_csv(f'{SAVE_DIR}/C_BCR_repertoire_MW.csv', index=False)
# Save donor-level data
bcr_full = pd.concat([donor_bcr_full(obs,'Liver'), donor_bcr_full(obs,'Blood')], ignore_index=True)
bcr_full.to_csv(f'{SAVE_DIR}/C_BCR_donor_level_full.csv', index=False)
print(f'\nSaved: C_BCR_repertoire_MW.csv, C_BCR_donor_level_full.csv')

SECTION C: BCR Repertoire Donor-Level Metrics

──────────────────────────────────────────────────────────────────────
LIVER — BCR Repertoire
──────────────────────────────────────────────────────────────────────
  NL (6d): BCR=194, clon=0.5051, sing=1.4%, IgM=45.9% IgG=30.6% IgA=13.4% IgD=10.1% sw=44.0% V3-23=7.8%
  IT (5d): BCR=122, clon=0.5334, sing=0.9%, IgM=44.4% IgG=35.9% IgA=16.7% IgD=3.0% sw=52.6% V3-23=12.4%
  IA (5d): BCR=278, clon=0.4381, sing=2.2%, IgM=56.8% IgG=24.9% IgA=14.0% IgD=4.3% sw=38.9% V3-23=8.5%
  AR (1d): BCR=59, clon=0.5676, sing=0.5%, IgM=50.8% IgG=22.0% IgA=25.4% IgD=1.7% sw=47.5% V3-23=6.8%
  CR: no BCR

  Mann-Whitney NL→IT (Liver):
      clonality: 0.51→0.53 (↑5.6%) p=0.4642 [19/30] | IA:p=0.5368 | NS
      pct_singleton: 1.35→0.95 (↓30.1%) p=0.4642 [20/30] | IA:p=0.4286 | NS
      pct_IgM: 45.91→44.39 (↓3.3%) p=0.9307 [16/30] | IA:p=0.6623 | NS
      pct_IgG: 30.56→35.88 (↑17.4%) p=0.3142 [21/30] | IA:p=0.9271 | NS
      pct_IgA: 13.39→16.74 (↑25.0%) p=0.3

In [24]:
# Cell 5: TCR REPERTOIRE — Donor-Level
print('='*70)
print('SECTION D: TCR Repertoire Donor-Level Metrics')
print('='*70)

def donor_tcr_full(obs_df, tissue_val):
    sub = obs_df[obs_df['tissue'] == tissue_val]
    rows = []
    for (stage, donor), grp in sub.groupby(['Stage', 'donor'], observed=True):
        n = len(grp)
        tcr = grp[grp['TCR_clone.id'].notna()]
        nt = len(tcr)
        if nt == 0:
            rows.append({'Stage':stage,'donor':donor,'tissue':tissue_val,
                         'n_tcr':0,'pct_tcr':0,
                         'clonality':np.nan,'pct_singleton':np.nan,
                         'top_clone':0,'n_unique_clones':0,
                         'top_vgene':'','top_vgene_pct':np.nan})
            continue
        cc = tcr['TCR_clone.id'].value_counts()
        clon = safe_clonality(cc)
        nu = len(cc); ns = (cc==1).sum()
        vg = tcr['TCR_v_gene.x'].value_counts()
        top_vg = vg.index[0] if len(vg)>0 else ''
        top_vg_pct = vg.iloc[0]/nt*100 if len(vg)>0 else 0
        rows.append({'Stage':stage,'donor':donor,'tissue':tissue_val,
                     'n_tcr':nt,'pct_tcr':nt/n*100,
                     'clonality':clon,'pct_singleton':ns/nu*100,
                     'top_clone':cc.max(),'n_unique_clones':nu,
                     'top_vgene':top_vg,'top_vgene_pct':top_vg_pct})
    return pd.DataFrame(rows)

tcr_results = []
for tissue_val in ['Liver','Blood']:
    df = donor_tcr_full(obs, tissue_val)
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — TCR Repertoire')
    print(f'{"─"*70}')
    for stage in ['NL','IT','IA','AR','CR']:
        s = df[(df['Stage']==stage) & (df['n_tcr']>0)]
        if len(s)==0:
            print(f'  {stage}: no TCR'); continue
        print(f'  {stage} ({len(s)}d): TCR={s.n_tcr.mean():.0f}, '
              f'clon={s.clonality.mean():.4f}, sing={s.pct_singleton.mean():.1f}%, '
              f'top_clone={s.top_clone.mean():.0f}, unique={s.n_unique_clones.mean():.0f}')

    nl = df[(df['Stage']=='NL') & (df['n_tcr']>0)]
    it = df[(df['Stage']=='IT') & (df['n_tcr']>0)]
    ia = df[(df['Stage']=='IA') & (df['n_tcr']>0)]
    if len(nl)>=2 and len(it)>=2:
        print(f'\n  Mann-Whitney NL→IT ({tissue_val}):')
        for m in ['clonality','pct_singleton','pct_tcr','top_clone','n_unique_clones']:
            r_it = mw_test(nl[m], it[m], m)
            r_ia = mw_test(nl[m], ia[m], m) if len(ia)>=2 else None
            if r_it:
                it_sig = r_it['p_value']<0.05
                ia_sig = r_ia['p_value']<0.05 if r_ia else False
                pattern = 'IT-spec' if it_sig and not ia_sig else \
                          'Chronic' if it_sig and ia_sig else \
                          'IA-emrg' if not it_sig and ia_sig else 'NS'
                ia_p = f'p={r_ia["p_value"]:.4f}' if r_ia else 'N/A'
                r_it['tissue']=tissue_val; r_it['pattern']=pattern
                tcr_results.append(r_it)
                print(f'    {r_it["sig"]} {m}: {r_it["NL_mean"]:.3f}→{r_it["IT_mean"]:.3f} '
                      f'({r_it["direction"]}{abs(r_it["pct_change"]):.1f}%) p={r_it["p_value"]:.4f} '
                      f'[{r_it["consistency"]}] | IA:{ia_p} | {pattern}')

pd.DataFrame(tcr_results).to_csv(f'{SAVE_DIR}/D_TCR_repertoire_MW.csv', index=False)
tcr_full = pd.concat([donor_tcr_full(obs,'Liver'), donor_tcr_full(obs,'Blood')], ignore_index=True)
tcr_full.to_csv(f'{SAVE_DIR}/D_TCR_donor_level_full.csv', index=False)
print(f'\nSaved: D_TCR_repertoire_MW.csv, D_TCR_donor_level_full.csv')

SECTION D: TCR Repertoire Donor-Level Metrics

──────────────────────────────────────────────────────────────────────
LIVER — TCR Repertoire
──────────────────────────────────────────────────────────────────────
  NL (6d): TCR=918, clon=0.5501, sing=0.8%, top_clone=135, unique=40517
  IT (6d): TCR=1449, clon=0.4874, sing=1.6%, top_clone=87, unique=40517
  IA (5d): TCR=3457, clon=0.3653, sing=3.9%, top_clone=98, unique=40517
  AR (1d): TCR=1767, clon=0.4228, sing=1.9%, top_clone=151, unique=40517
  CR: no TCR

  Mann-Whitney NL→IT (Liver):
      clonality: 0.550→0.487 (↓11.4%) p=0.1320 [28/36] | IA:p=0.0043 | IA-emrg
      pct_singleton: 0.806→1.590 (↑97.2%) p=0.3939 [24/36] | IA:p=0.0303 | IA-emrg
    † pct_tcr: 18.806→35.594 (↑89.3%) p=0.0649 [30/36] | IA:p=0.0519 | NS
      top_clone: 135.333→87.000 (↓35.7%) p=0.7483 [21/36] | IA:p=0.9307 | NS
      n_unique_clones: 40517.000→40517.000 (↓0.0%) p=1.0000 [36/36] | IA:p=1.0000 | NS

──────────────────────────────────────────────────────

In [25]:
# Cell 6: COMPREHENSIVE SUMMARY TABLE
print('='*70)
print('COMPREHENSIVE SUMMARY: ALL SIGNIFICANT + TREND FINDINGS')
print('='*70)

all_results = []
for r in results_a:  # subcluster proportions
    r['section'] = 'A_subcluster_prop'
    all_results.append(r)
for r in results_b:  # differentiation ratios
    r['section'] = 'B_diff_ratio'
    all_results.append(r)
for r in bcr_results:  # BCR repertoire
    r['section'] = 'C_BCR_repertoire'
    all_results.append(r)
for r in tcr_results:  # TCR repertoire
    r['section'] = 'D_TCR_repertoire'
    all_results.append(r)

all_df = pd.DataFrame(all_results)

# Show significant and trend findings
sig_df = all_df[all_df['p_value'] < 0.10].sort_values('p_value')
print(f'\nFindings with p < 0.10: {len(sig_df)}')
print(f'Findings with p < 0.05: {len(all_df[all_df["p_value"]<0.05])}')
print()
for _, r in sig_df.iterrows():
    print(f'{r["sig"]} [{r["section"]}] {r["tissue"]}/{r["metric"]}: '
          f'NL={r["NL_mean"]:.2f}→IT={r["IT_mean"]:.2f} '
          f'({r["direction"]}{abs(r["pct_change"]):.1f}%) '
          f'p={r["p_value"]:.4f} [{r["consistency"]}] | {r["pattern"]}')

all_df.to_csv(f'{SAVE_DIR}/MASTER_all_MW_results.csv', index=False)
sig_df.to_csv(f'{SAVE_DIR}/MASTER_significant_results.csv', index=False)
print(f'\nSaved: MASTER_all_MW_results.csv ({len(all_df)} tests)')
print(f'Saved: MASTER_significant_results.csv ({len(sig_df)} findings)')

# IT-specific findings
it_spec = all_df[all_df['pattern'].str.contains('IT-spec', na=False)]
print(f'\n--- IT-Specific findings: {len(it_spec)} ---')
for _, r in it_spec.iterrows():
    print(f'  {r["sig"]} {r["tissue"]}/{r["metric"]}: p={r["p_value"]:.4f} [{r["consistency"]}]')

print(f'\n✅ All C12 analysis complete. Results in: {SAVE_DIR}')

COMPREHENSIVE SUMMARY: ALL SIGNIFICANT + TREND FINDINGS

Findings with p < 0.10: 7
Findings with p < 0.05: 4

★ [C_BCR_repertoire] Blood/top_clone: NL=1.00→IT=3.00 (↑200.0%) p=0.0056 [25/25] | Chronic
★ [A_subcluster_prop] Liver/B_c04-COCH: NL=1.31→IT=7.06 (↑437.8%) p=0.0135 [29/30] | Chronic-persistent
★ [C_BCR_repertoire] Blood/pct_IgD: NL=17.71→IT=2.23 (↓87.4%) p=0.0159 [24/25] | Chronic
★ [D_TCR_repertoire] Blood/clonality: NL=0.59→IT=0.35 (↓39.9%) p=0.0317 [23/25] | IT-spec
† [A_subcluster_prop] Liver/B_c07-FCRL5: NL=12.97→IT=28.22 (↑117.7%) p=0.0519 [26/30] | NS
† [D_TCR_repertoire] Liver/pct_tcr: NL=18.81→IT=35.59 (↑89.3%) p=0.0649 [30/36] | NS
† [A_subcluster_prop] Liver/plasmaB_c03-MKI67: NL=8.19→IT=1.40 (↓82.9%) p=0.0823 [25/30] | NS

Saved: MASTER_all_MW_results.csv (56 tests)
Saved: MASTER_significant_results.csv (7 findings)

--- IT-Specific findings: 1 ---
  ★ Blood/clonality: p=0.0317 [23/25]

✅ All C12 analysis complete. Results in: /content/drive/MyDrive/ITLAS/results/

In [ ]:
# SDC1 Grouped Analysis
'''
만약 SDC1 collapse가 IT/IA/CR 모두에서 일어나고, NL과 AR에서는 유지된다면 → 이것은 "chronic HBV 공통 특성"이지 특정 phase의 문제가 아님 → (NL+AR) vs (IT+IA+CR) 비교가 올바른 프레임
'''

In [26]:
# Cell: SDC1 Collapse — All Pairwise + Grouped Analysis
# Run after C12 notebook (uses obs, ratio_df from Cell 3)
from scipy.stats import mannwhitneyu
import numpy as np, pandas as pd

# Recalculate ratio_df if not available
b_plasma = obs[obs['major_lineage'].isin(['B', 'PlasmaB'])].copy()
obs['donor'] = obs['sample'].astype(str).str.split('_').str[1]

ratio_rows = []
for (stage, tissue, donor), grp in b_plasma.groupby(['Stage', 'tissue', 'donor'], observed=True):
    n_b = (grp['major_lineage'] == 'B').sum()
    n_pb = (grp['major_lineage'] == 'PlasmaB').sum()
    n_sdc1 = (grp['gut2021_subcluster_v2'] == 'plasmaB_c01-SDC1').sum()
    total = n_b + n_pb
    if total < 5:
        continue
    ratio_rows.append({
        'Stage': stage, 'tissue': tissue, 'donor': donor,
        'n_SDC1': n_sdc1, 'total_BP': total,
        'SDC1_pct': n_sdc1 / total * 100,
        'PlasmaB_pct': n_pb / total * 100,
    })
ratio_df = pd.DataFrame(ratio_rows)

def mw_with_consistency(a, b, label=''):
    a = a.dropna(); b = b.dropna()
    if len(a) < 2 or len(b) < 2:
        return None
    stat, p = mannwhitneyu(a, b, alternative='two-sided')
    am, bm = a.mean(), b.mean()
    d = '↑' if bm > am else '↓'
    pct = ((bm - am) / am * 100) if am != 0 else float('inf')
    pairs_t = 0; pairs_c = 0
    for av in a:
        for bv in b:
            pairs_t += 1
            if (bm > am and bv > av) or (bm <= am and bv <= av):
                pairs_c += 1
    sig = '★' if p < 0.05 else '†' if p < 0.10 else ' '
    return {'label': label, 'grpA_n': len(a), 'grpB_n': len(b),
            'grpA_mean': am, 'grpB_mean': bm,
            'direction': d, 'pct_change': pct, 'p': p,
            'consistency': f'{pairs_c}/{pairs_t}', 'sig': sig}

print('='*70)
print('SDC1 COLLAPSE: ALL PAIRWISE COMPARISONS')
print('='*70)

for tissue_val in ['Liver', 'Blood']:
    t = ratio_df[ratio_df['tissue'] == tissue_val]
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()} — SDC1 % of B+PlasmaB')
    print(f'{"─"*70}')

    # Show raw donor values per stage
    for stage in ['NL', 'IT', 'IA', 'AR', 'CR']:
        s = t[t['Stage'] == stage]
        if len(s) == 0:
            print(f'  {stage}: no data')
            continue
        vals = s['SDC1_pct'].values
        print(f'  {stage} (n={len(s)}): mean={s.SDC1_pct.mean():.1f}%, '
              f'donors=[{", ".join(f"{v:.1f}" for v in vals)}]')

    # All pairwise comparisons
    stages = ['NL', 'IT', 'IA', 'AR', 'CR']
    print(f'\n  All pairwise Mann-Whitney (SDC1_pct):')
    for i, s1 in enumerate(stages):
        for s2 in stages[i+1:]:
            a = t[t['Stage'] == s1]['SDC1_pct']
            b = t[t['Stage'] == s2]['SDC1_pct']
            r = mw_with_consistency(a, b, f'{s1}→{s2}')
            if r:
                print(f'    {r["sig"]} {s1}→{s2}: {r["grpA_mean"]:.1f}%→{r["grpB_mean"]:.1f}% '
                      f'({r["direction"]}{abs(r["pct_change"]):.1f}%) p={r["p"]:.4f} [{r["consistency"]}]')
            else:
                print(f'      {s1}→{s2}: insufficient data')

print(f'\n{"="*70}')
print('SDC1 COLLAPSE: GROUPED ANALYSIS')
print('  Chronic group: IT + IA + CR (vertical infection → chronic)')
print('  Non-chronic group: NL + AR (never chronic / acute resolved)')
print('='*70)

for tissue_val in ['Liver', 'Blood']:
    t = ratio_df[ratio_df['tissue'] == tissue_val]
    print(f'\n{"─"*70}')
    print(f'{tissue_val.upper()}')
    print(f'{"─"*70}')

    # Group: chronic (IT+IA+CR) vs non-chronic (NL+AR)
    chronic = t[t['Stage'].isin(['IT', 'IA', 'CR'])]['SDC1_pct']
    non_chronic = t[t['Stage'].isin(['NL', 'AR'])]['SDC1_pct']

    print(f'  Non-chronic (NL+AR): n={len(non_chronic)}, mean={non_chronic.mean():.1f}%, '
          f'donors=[{", ".join(f"{v:.1f}" for v in non_chronic.values)}]')
    print(f'  Chronic (IT+IA+CR):  n={len(chronic)}, mean={chronic.mean():.1f}%, '
          f'donors=[{", ".join(f"{v:.1f}" for v in chronic.values)}]')

    r = mw_with_consistency(non_chronic, chronic, 'NL+AR vs IT+IA+CR')
    if r:
        print(f'\n  {r["sig"]} (NL+AR) vs (IT+IA+CR): {r["grpA_mean"]:.1f}%→{r["grpB_mean"]:.1f}% '
              f'({r["direction"]}{abs(r["pct_change"]):.1f}%) p={r["p"]:.4f} [{r["consistency"]}]')

    # Also test: NL vs (IT+IA+CR) — exclude AR
    chronic_no_ar = t[t['Stage'].isin(['IT', 'IA', 'CR'])]['SDC1_pct']
    nl_only = t[t['Stage'] == 'NL']['SDC1_pct']
    r2 = mw_with_consistency(nl_only, chronic_no_ar, 'NL vs IT+IA+CR')
    if r2:
        print(f'  {r2["sig"]} NL vs (IT+IA+CR): {r2["grpA_mean"]:.1f}%→{r2["grpB_mean"]:.1f}% '
              f'({r2["direction"]}{abs(r2["pct_change"]):.1f}%) p={r2["p"]:.4f} [{r2["consistency"]}]')

    # Test: NL vs AR (should be similar if AR truly resolved)
    ar_only = t[t['Stage'] == 'AR']['SDC1_pct']
    r3 = mw_with_consistency(nl_only, ar_only, 'NL vs AR')
    if r3:
        print(f'  {r3["sig"]} NL vs AR: {r3["grpA_mean"]:.1f}%→{r3["grpB_mean"]:.1f}% '
              f'({r3["direction"]}{abs(r3["pct_change"]):.1f}%) p={r3["p"]:.4f} [{r3["consistency"]}]')
    else:
        print(f'  NL vs AR: insufficient data (AR n={len(ar_only)})')

    # IA vs AR
    ia_only = t[t['Stage'] == 'IA']['SDC1_pct']
    r4 = mw_with_consistency(ia_only, ar_only, 'IA vs AR')
    if r4:
        print(f'  {r4["sig"]} IA vs AR: {r4["grpA_mean"]:.1f}%→{r4["grpB_mean"]:.1f}% '
              f'({r4["direction"]}{abs(r4["pct_change"]):.1f}%) p={r4["p"]:.4f} [{r4["consistency"]}]')
    else:
        print(f'  IA vs AR: insufficient data')

SDC1 COLLAPSE: ALL PAIRWISE COMPARISONS

──────────────────────────────────────────────────────────────────────
LIVER — SDC1 % of B+PlasmaB
──────────────────────────────────────────────────────────────────────
  NL (n=6): mean=32.5%, donors=[2.0, 77.0, 31.6, 26.0, 55.9, 2.3]
  IT (n=5): mean=26.0%, donors=[40.0, 11.5, 14.9, 17.2, 46.3]
  IA (n=5): mean=5.0%, donors=[0.0, 9.2, 2.2, 6.6, 6.8]
  AR (n=3): mean=7.8%, donors=[6.0, 3.7, 13.8]
  CR (n=3): mean=6.0%, donors=[9.6, 7.3, 1.1]

  All pairwise Mann-Whitney (SDC1_pct):
      NL→IT: 32.5%→26.0% (↓19.9%) p=0.9307 [16/30]
      NL→IA: 32.5%→5.0% (↓84.7%) p=0.1775 [23/30]
      NL→AR: 32.5%→7.8% (↓75.9%) p=0.5476 [12/18]
      NL→CR: 32.5%→6.0% (↓81.4%) p=0.2619 [14/18]
    ★ IT→IA: 26.0%→5.0% (↓80.8%) p=0.0079 [25/25]
    † IT→AR: 26.0%→7.8% (↓69.9%) p=0.0714 [14/15]
    ★ IT→CR: 26.0%→6.0% (↓76.8%) p=0.0357 [15/15]
      IA→AR: 5.0%→7.8% (↑57.4%) p=0.7857 [9/15]
      IA→CR: 5.0%→6.0% (↑21.3%) p=0.5714 [10/15]
      AR→CR: 7.8%→6.0% 

In [27]:
# Same analysis for PlasmaB_pct (total PlasmaB ratio)
print(f'\n{"="*70}')
print('SAME GROUPED ANALYSIS FOR: PlasmaB_pct (total PlasmaB ratio)')
print('='*70)

for tissue_val in ['Liver', 'Blood']:
    t = ratio_df[ratio_df['tissue'] == tissue_val]
    print(f'\n  {tissue_val.upper()}:')

    for metric in ['PlasmaB_pct']:
        chronic = t[t['Stage'].isin(['IT', 'IA', 'CR'])][metric]
        non_chronic = t[t['Stage'].isin(['NL', 'AR'])][metric]
        r = mw_with_consistency(non_chronic, chronic, f'(NL+AR) vs (IT+IA+CR) {metric}')
        if r:
            print(f'  {r["sig"]} (NL+AR) vs (IT+IA+CR) {metric}: '
                  f'{r["grpA_mean"]:.1f}%→{r["grpB_mean"]:.1f}% '
                  f'({r["direction"]}{abs(r["pct_change"]):.1f}%) p={r["p"]:.4f} [{r["consistency"]}]')

        nl = t[t['Stage'] == 'NL'][metric]
        chronic_all = t[t['Stage'].isin(['IT', 'IA', 'CR'])][metric]
        r2 = mw_with_consistency(nl, chronic_all, f'NL vs (IT+IA+CR) {metric}')
        if r2:
            print(f'  {r2["sig"]} NL vs (IT+IA+CR) {metric}: '
                  f'{r2["grpA_mean"]:.1f}%→{r2["grpB_mean"]:.1f}% '
                  f'({r2["direction"]}{abs(r2["pct_change"]):.1f}%) p={r2["p"]:.4f} [{r2["consistency"]}]')


SAME GROUPED ANALYSIS FOR: PlasmaB_pct (total PlasmaB ratio)

  LIVER:
    (NL+AR) vs (IT+IA+CR) PlasmaB_pct: 36.1%→20.4% (↓43.3%) p=0.3498 [73/117]
    NL vs (IT+IA+CR) PlasmaB_pct: 44.1%→20.4% (↓53.6%) p=0.3229 [51/78]

  BLOOD:
    (NL+AR) vs (IT+IA+CR) PlasmaB_pct: 14.3%→6.9% (↓51.9%) p=0.3054 [62/96]
    NL vs (IT+IA+CR) PlasmaB_pct: 17.0%→6.9% (↓59.5%) p=0.7990 [33/60]


In [28]:
# Save
import os
SAVE_DIR = '/content/drive/MyDrive/ITLAS/results/version18-analysis-v2/BCR_TCR'
os.makedirs(SAVE_DIR, exist_ok=True)
ratio_df.to_csv(f'{SAVE_DIR}/SDC1_donor_level_all.csv', index=False)
print(f'\nSaved: SDC1_donor_level_all.csv')



Saved: SDC1_donor_level_all.csv
